# FusionMatch — Phase 5: ONNX Export, INT8 Quantization & FastAPI Serving

This notebook demonstrates:
1. **ONNX Export**: Converting PyTorch `FusionMatchModel` to ONNX graph representation with dynamic batch axes.
2. **Dynamic INT8 Quantization**: Quantizing model weights for accelerated CPU inference (~4x smaller, ~2.5x faster).
3. **Inference Latency Profiling**: Measuring P50, P95, and P99 latency against our $< 15$ ms SLA budget.
4. **FastAPI Client Testing**: Querying the REST API endpoints (`/health`, `/v1/check`, `/v1/check/batch`).

In [ ]:
import sys
import io
import base64
import time
import statistics
from pathlib import Path
import numpy as np
from PIL import Image
import onnxruntime as ort
from fastapi.testclient import TestClient

# Ensure project root is on sys.path
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.models import FusionMatchModel
from src.export import export_to_onnx, quantize_dynamic_int8
from src.serving.main import app
from src.utils.seed import seed_everything

seed_everything(42)
print(f"ONNX Runtime version: {ort.__version__}")

## 1. Export PyTorch Model to ONNX & Quantize to INT8

In [ ]:
onnx_dir = project_root / "artifacts" / "onnx"
onnx_dir.mkdir(parents=True, exist_ok=True)

fp32_path = onnx_dir / "fusion_match_fp32.onnx"
int8_path = onnx_dir / "fusion_match_int8.onnx"

model = FusionMatchModel(use_mock=True, embed_dim=256)
export_to_onnx(model, fp32_path, use_mock=True)
quantize_dynamic_int8(fp32_path, int8_path)

fp32_size_mb = fp32_path.stat().st_size / (1024 * 1024)
int8_size_mb = int8_path.stat().st_size / (1024 * 1024)
print("\n=== ONNX STORAGE COMPRESSION ===")
print(f"FP32 ONNX Model: {fp32_size_mb:.2f} MB")
print(f"INT8 ONNX Model: {int8_size_mb:.2f} MB (Compression: {fp32_size_mb/int8_size_mb:.1f}x)")

## 2. Benchmark CPU Inference Latency (P50 / P95 / P99)

We profile sequential single-query latency over 100 requests to verify that CPU inference meets the $\le 15\text{ ms}$ budget.

In [ ]:
client = TestClient(app)

# Create sample Base64 image
img = Image.new("RGB", (224, 224), color=(80, 140, 200))
buf = io.BytesIO()
img.save(buf, format="JPEG")
img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")

payload = {
    "image_base64": img_b64,
    "title": "AmazonBasics Ergonomic Wireless Optical Mouse with USB Nano Receiver",
    "category": "ELECTRONICS",
    "top_k": 5,
}

# Warm-up (10 requests) 
for _ in range(10):
    client.post("/v1/check", json=payload)

# Benchmark (100 requests)
latencies_ms = []
for _ in range(100):
    t0 = time.perf_counter()
    resp = client.post("/v1/check", json=payload)
    latencies_ms.append((time.perf_counter() - t0) * 1000.0)

latencies_ms.sort()
p50 = statistics.median(latencies_ms)
p95 = latencies_ms[int(0.95 * len(latencies_ms))]
p99 = latencies_ms[int(0.99 * len(latencies_ms))]

print("=== INFERENCE LATENCY BENCHMARK ===")
print(f"P50 Latency: {p50:.2f} ms")
print(f"P95 Latency: {p95:.2f} ms (Target SLA: < 15.0 ms)")
print(f"P99 Latency: {p99:.2f} ms")

## 3. Test REST API Endpoints

In [ ]:
# Health Check
health_resp = client.get("/health").json()
print("Health Check Response:", health_resp)

# Single Item Check
single_resp = client.post("/v1/check", json=payload).json()
print("\nSingle Check Result:")
print(f"Is Duplicate:    {single_resp['is_duplicate']}")
print(f"Threshold Used:  {single_resp['threshold_used']}")
print(f"Gate Weights:    {single_resp['gate_weights']}")
print(f"Top Candidate:   {single_resp['candidates'][0] if single_resp['candidates'] else 'None'}")

# Batch Check
batch_payload = {
    "items": [
        {"item_id": "sku_1", "image_base64": img_b64, "title": "Wireless Mouse", "top_k": 3},
        {"item_id": "sku_2", "image_base64": img_b64, "title": "Gaming Keyboard", "top_k": 3},
    ],
    "top_k": 3,
}
batch_resp = client.post("/v1/check/batch", json=batch_payload).json()
print(f"\nBatch Check Results Count: {len(batch_resp['results'])}")